# Silver: Batidas de ponto

Preparação dos dados da bronze antes de salvar na silver. Sem cálculos de negócio: só tipagem, padronização, deduplicação e validação.

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
SOURCE_TABLE = "projetos.bronze.ponto"
TARGET_TABLE = "projetos.silver.tab_batidas_de_ponto"

REQUIRED_COLUMNS = [
    "id_batida", "id_cracha", "worker_id"
]

In [0]:
df = spark.table(SOURCE_TABLE)

## Data Quality

In [0]:
rename_map = {
    "ID_Batida": "id_batida",
    "ID_Cracha": "id_cracha",
    "Worker_ID": "worker_id",
    "Data_Batida": "data_batida",
    "Hora_Batida": "hora_batida",
    "Data_Hora_UTC": "data_hora_utc",
    "Tipo_Batida": "tipo_batida",
    "Sequencia_Dia": "sequencia_dia",
    "Origem": "origem",
    "Equipamento_ID": "equipamento_id",
    "Localizacao": "localizacao",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Status": "status",
    "Justificativa": "justificativa",
    "Fuso_Horario": "fuso_horario",
    "_gdrive_metadata": "_gdrive_metadata",
    "_file_metadata": "_file_metadata",
}

existing_columns = set(df.columns)

for old_name, new_name in rename_map.items():
    if old_name in existing_columns and old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

Padronizar textos: remove espaços e transforma "null" / vazio em NULL de verdade

In [0]:
NULL_TOKENS = ["null", "NULL", "Null", ""]

string_cols = [f.name for f in df.schema.fields
               if f.dataType.typeName() == "string" and not f.name.startswith("_")]

for c in string_cols:
    limpo = F.trim(F.col(c))
    df = df.withColumn(c, F.when(limpo.isin(NULL_TOKENS) | limpo.isNull(), F.lit(None))
                           .otherwise(limpo))


Tipagem

In [0]:
df = df.withColumn('data_batida', F.to_date('data_batida', 'yyyy-MM-dd'))
df = df.withColumn('data_hora_utc', F.to_timestamp(F.concat(df.data_batida, F.lit(' '), df.hora_batida), 'yyyy-MM-dd HH:mm:ss'))

Metadados de ingestão

In [0]:
df = df.withColumn('source_file', df._file_metadata["file_name"])
df = df.withColumn('file_modification_time', df._file_metadata["file_modification_time"])
df = df.drop("_file_metadata", '_gdrive_metadata')
df = df.dropDuplicates()

Validação: falha cedo em vez de gravar lixo na silver

In [0]:
colunas_faltando = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if colunas_faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {colunas_faltando}")

chaves_nulas = df.where(F.col("worker_id").isNull()).count()
if chaves_nulas > 0:
    raise ValueError(f"{chaves_nulas} registros sem worker_id")

## Save

In [0]:
source = df 
target_table = DeltaTable.forName(spark, TARGET_TABLE)

target_table.alias("target")\
    .merge(source.alias("source"), condition="target.id_batida = source.id_batida AND target.file_modification_time > source.file_modification_time")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()